# Импорты

In [8]:
from ultralytics import YOLO
from detection import DetectorYOLO
from photometric_augm import Exposition

In [2]:
%load_ext autoreload
%autoreload 2

# Обучение

In [ ]:
custom_transforms = [
    Exposition(
        crf_path="crf.yaml",
        p=0.3,
    )
]

In [ ]:
model = YOLO('yolo11s.pt')

model.train(
    data='data.yaml',
    epochs=100,
    imgsz=960,
    device='cuda',
    batch=4,
    nbs=32,
    workers=4,
    optimizer='AdamW',
    seed=1,
    cos_lr=True,
    lr0=1e-3,
    hsv_s=0.35,
    hsv_v=0.25,
    degrees=10,
    translate=0.03,
    perspective=0.0003,
    scale=0.3,
    shear=10,
    fliplr=0.0,
    erasing=0.0,
    mosaic=0.5,
    cutmix=0.5,
    close_mosaic=5,
    save_crop=True,
    nms=True,
    augmentations=custom_transforms,
)

# Детекция

In [3]:
model_path = 'weights/best.pt'
tracker = 'bytetrack.yaml'
save_root_dir = 'saved_crops'
img_path = 'your_img_dir_or_filepath'
video_path = 'videos/1.mp4'

## Изображение

In [ ]:
det = DetectorYOLO(
    model_path=model_path,
    tracker=tracker,
    save_root_dir=save_root_dir,
    rotate=True,
    frame_skip=3,
    q_crops=5,
    imgsz=960,
    max_det=200,
)

for fname, img, xyxys, confs, clss in det.image_detection(
    img_path=img_path,
    result_show=True
):
    # print(fname, img, xyxys, confs, clss)

    input('Нажмите Enter, чтобы продолжить')

WARNING ⚠️ 'source' is missing. Using 'source=/mnt/e/LentaTech/.venv/lib/python3.12/site-packages/ultralytics/assets'.


[ WARN:0@968.475] global loadsave.cpp:241 findDecoder imread_('your_img_dir_or_filepath'): can't open/read file: check file path/integrity


# Видео

In [4]:
frame_diff = 100

In [19]:
det = DetectorYOLO(
    model_path =model_path,
    tracker=tracker,
    save_root_dir=save_root_dir,
    rotate=True,
    frame_skip=1,
    q_crops=5,
    conf=0.5,
    imgsz=960,
    max_det=200,
    save_tracks_csv=True,
    tracks_csv_path="tracks.csv",
)

for frame_idx, img, grade_d in det.video_detection(
    video_path=video_path,
    video_save=True,
    video_show=True
):
    if frame_idx % det.frame_skip == 0:
        for tr_id, vals in list(grade_d.items()):
            if len(vals) > 0:
                frame_last = max(vals, key= lambda x: x['frame_idx'])['frame_idx']
            if frame_idx - frame_last >= frame_diff:    

                # OCR Logic

                det.clear_track(tr_id)